# Analiz 4: Kaza ve Arıza İlişkisi — Tam Kanıt Zinciri

**Hedef:** Kaza ↔ Arıza ilişkisini istatistiksel testlerle kanıtlamak.

**Yöntem:** Şartlama yok, veri konuşur. Her iddiayı kanıtla destekle.

---

## 1. Veri Yükleme

In [1]:
import pandas as pd
import numpy as np
import json
import plotly.express as px
import plotly.graph_objects as go
from datetime import timedelta
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Veri yukleme
df_ariza = pd.read_csv('../panel_data/temiz_veri/ariza_model.csv')
df_ariza['OLAYTARIHI'] = pd.to_datetime(df_ariza['OLAYTARIHI'], format='mixed')

with open('../panel_data/kaza_geojson.json', 'r', encoding='utf-8') as f:
    kaza_data = json.load(f)

kaza_list = []
for feature in kaza_data['features']:
    props = feature['properties']
    props['lon'] = feature['geometry']['coordinates'][0]
    props['lat'] = feature['geometry']['coordinates'][1]
    kaza_list.append(props)

df_kaza = pd.DataFrame(kaza_list)
df_kaza['kazasaat'] = pd.to_datetime(df_kaza['kazasaat'], format='mixed')
df_kaza = df_kaza.dropna(subset=['kapino', 'kazasaat'])

print(f'Yuklenen Ariza Sayisi: {len(df_ariza):,}')
print(f'Yuklenen Kaza Sayisi:  {len(df_kaza):,}')
print(f'Ariza tarih araligi: {df_ariza["OLAYTARIHI"].min()} -> {df_ariza["OLAYTARIHI"].max()}')
print(f'Kaza tarih araligi:  {df_kaza["kazasaat"].min()} -> {df_kaza["kazasaat"].max()}')
print(f'Ortak araclar (KAPINO): {len(set(df_ariza["KAPINO"]) & set(df_kaza["kapino"])):,}')


Yuklenen Ariza Sayisi: 58,559
Yuklenen Kaza Sayisi:  2,900
Ariza tarih araligi: 2025-01-01 00:30:56 -> 2025-06-30 23:29:56
Kaza tarih araligi:  2025-01-01 10:25:00 -> 2025-06-30 22:41:00
Ortak araclar (KAPINO): 1,420


---

## 1.5. ÖHO/KOOP Filtresi (Kritik Veri Düzeltmesi)
İETT yalnızca kendi filosunun arıza kaydını tutuyor. ÖHO/Kooperatif araçları (C/D-prefix + kısmi B-prefix) kaza datasında VAR ama arıza datasında YOK. Adil karşılaştırma için filtre uygulanıyor.

In [2]:
# BOLUM 1.5: OHO/KOOP Aracları Filtresi (Kayit Disilik Duzeltmesi)
# IETT yalnizca kendi filosunun ariza kaydini tutuyor.
# OHO/Kooperatif araclari (genelde C-prefix, kismi B-prefix, D-prefix) kaza datasinda var
# ama arizada YOK. Analiz icin sadece her iki datada da var olan araclari tutuyoruz.

ariza_kapinolar = set(df_ariza['KAPINO'].unique())
df_kaza_full = df_kaza.copy()  # orijinali sakla
df_kaza = df_kaza[df_kaza['kapino'].isin(ariza_kapinolar)].copy()

print('=== OHO/KOOP FILTRESI ===')
print(f'Orijinal kaza sayisi: {len(df_kaza_full):,}')
print(f'Filtrelenmis kaza:    {len(df_kaza):,}')
print(f'Cikarilan kaza:       {len(df_kaza_full)-len(df_kaza):,}')
print()

# Plaka oneki dagilimi
print('=== ONEKE GORE FILTRELEME ETKISI ===')
print(f'{"Onek":6s}  Orijinal  Filtreli  Cikarilan')
for onek in sorted(set([k[0] for k in df_kaza_full["kapino"]])):
    orig = (df_kaza_full['kapino'].str[0] == onek).sum()
    filt = (df_kaza['kapino'].str[0] == onek).sum()
    print(f'{onek:6s}  {orig:8d}  {filt:8d}  {orig-filt:9d}')

print()
print(f'Kalan benzersiz arac: {df_kaza["kapino"].nunique():,}')
print(f'Bunlar IETT kendi filosu — ariza kaydi tutuluyor.')


=== OHO/KOOP FILTRESI ===
Orijinal kaza sayisi: 2,900
Filtrelenmis kaza:    1,967
Cikarilan kaza:       933

=== ONEKE GORE FILTRELEME ETKISI ===
Onek    Orijinal  Filtreli  Cikarilan
A            415       139        276
B            510       303        207
C            423         0        423
D             27         0         27
K            356       356          0
M            459       459          0
O            636       636          0
T             74        74          0

Kalan benzersiz arac: 1,420
Bunlar IETT kendi filosu — ariza kaydi tutuluyor.


---

## 2. Baseline Arıza Oranı Hesabı
Kaza-olmayan dönemlerde araç başına günlük arıza oranı.
Bunu ölçmeden "kaza sonrası fırtına" iddiası kanıtlanamaz.

In [3]:
# BOLUM 2: Baseline Ariza Orani
# Soru: Kaza-olmayan donemlerde araç başına günlük arıza oranı nedir?
# Bunu olçmeden "kaza sonrası fırtına" iddiası kanıtlanmaz.

# Tum arac × gun matrisi: kaç ariza?
ariza_gun = df_ariza.groupby(['KAPINO', df_ariza['OLAYTARIHI'].dt.date]).size().reset_index(name='ariza_sayisi')

# Veri penceresi
toplam_gun = (df_ariza['OLAYTARIHI'].max() - df_ariza['OLAYTARIHI'].min()).days
toplam_arac = df_ariza['KAPINO'].nunique()
toplam_ariza = len(df_ariza)

# Genel ortalama: arac × gun başına ariza sayısı (zero-inflated dahil)
ariza_orani_global = toplam_ariza / (toplam_arac * toplam_gun)

# Kaza yapan araçların kaza-olmayan donemleri (tum penceresi - kaza ±30g pencereler)
print('Baseline hesaplanıyor...')
kaza_pencereleri = []
for _, row in df_kaza.iterrows():
    kapino = row['kapino']
    kaza_t = row['kazasaat']
    kaza_pencereleri.append({
        'KAPINO': kapino,
        'baslangic': kaza_t - timedelta(days=2),
        'bitis': kaza_t + timedelta(days=30)
    })
kaza_pen_df = pd.DataFrame(kaza_pencereleri)

# Her ariza icin kaza-yakin mi kontrol et
def is_in_kaza_window(row):
    pencere = kaza_pen_df[kaza_pen_df['KAPINO'] == row['KAPINO']]
    if len(pencere) == 0:
        return False
    return any((pencere['baslangic'] <= row['OLAYTARIHI']) & (row['OLAYTARIHI'] <= pencere['bitis']))

# Hızlı vectorize
df_ariza_kaza_iliski = df_ariza[['KAPINO', 'OLAYTARIHI']].merge(
    df_kaza[['kapino', 'kazasaat']], left_on='KAPINO', right_on='kapino', how='left'
)
df_ariza_kaza_iliski['gun_farki'] = (
    df_ariza_kaza_iliski['OLAYTARIHI'] - df_ariza_kaza_iliski['kazasaat']
).dt.total_seconds() / 86400

# Bir ariza kaza penceresi icindeyse (-2 ile +30 gun): kaza-yakin
arizada_kaza_var = df_ariza_kaza_iliski.groupby(level=0).apply(
    lambda g: ((g['gun_farki'] >= -2) & (g['gun_farki'] <= 30)).any()
)

# Daha hızlı: arıza index'i bazlı flag
df_ariza_idx = df_ariza.copy().reset_index(drop=True)
df_ariza_idx['kaza_yakin'] = False

# Her kaza için arıza penceresini işaretle
for _, row in df_kaza.iterrows():
    kapino = row['kapino']
    kaza_t = row['kazasaat']
    mask = (
        (df_ariza_idx['KAPINO'] == kapino) &
        (df_ariza_idx['OLAYTARIHI'] >= kaza_t - timedelta(days=2)) &
        (df_ariza_idx['OLAYTARIHI'] <= kaza_t + timedelta(days=30))
    )
    df_ariza_idx.loc[mask, 'kaza_yakin'] = True

baseline_ariza = df_ariza_idx[~df_ariza_idx['kaza_yakin']]
kaza_yakin_ariza = df_ariza_idx[df_ariza_idx['kaza_yakin']]

# Baseline gun-arac sayisini hesapla
baseline_aracgun = toplam_arac * toplam_gun - len(kaza_pen_df) * 32
baseline_orani = len(baseline_ariza) / baseline_aracgun

print()
print('=== TEMEL ISTATISTIKLER ===')
print(f'Toplam veri penceresi: {toplam_gun} gun')
print(f'Toplam arac: {toplam_arac:,}')
print(f'Toplam ariza: {toplam_ariza:,}')
print(f'Genel orani (her arac × gun): {ariza_orani_global:.5f} ariza/arac/gun')
print()
print('=== BASELINE vs KAZA-YAKIN ===')
print(f'Baseline ariza: {len(baseline_ariza):,} ({len(baseline_ariza)/toplam_ariza*100:.1f}% tum arizalardan)')
print(f'Kaza-yakin ariza (-2..+30g): {len(kaza_yakin_ariza):,} ({len(kaza_yakin_ariza)/toplam_ariza*100:.1f}%)')
print(f'Kaza penceresi sayısı: {len(kaza_pen_df):,}  (her biri 32 gun)')
print(f'Baseline arac×gun: {baseline_aracgun:,}')
print(f'Baseline orani: {baseline_orani:.5f} ariza/arac/gun')


Baseline hesaplanıyor...

=== TEMEL ISTATISTIKLER ===
Toplam veri penceresi: 180 gun
Toplam arac: 3,509
Toplam ariza: 58,559
Genel orani (her arac × gun): 0.09271 ariza/arac/gun

=== BASELINE vs KAZA-YAKIN ===
Baseline ariza: 53,382 (91.2% tum arizalardan)
Kaza-yakin ariza (-2..+30g): 5,177 (8.8%)
Kaza penceresi sayısı: 1,967  (her biri 32 gun)
Baseline arac×gun: 568,676
Baseline orani: 0.09387 ariza/arac/gun


---

## 3. Kaza Sonrası Fırtına Testi (Poisson Rate Ratio)
Kaza sonrası 30 gün arıza oranı baseline'a göre yüksek mi?
Test: Poisson rate ratio + %95 CI + tek-tarafli z-test.

In [4]:
# BOLUM 3: Kaza Sonrasi Firtina (Poisson Rate Ratio)
# Soru: Kaza sonrasi 30 gun ariza orani baseline'a gore yuksek mi?
# Test: Poisson rate ratio + %95 guven araligi

# Her kazanin 30 gunluk pencerede ariza sayisi
post_kaza_ariza = []
for _, row in df_kaza.iterrows():
    kapino = row['kapino']
    kaza_t = row['kazasaat']
    n_ariza = (
        (df_ariza['KAPINO'] == kapino) &
        (df_ariza['OLAYTARIHI'] > kaza_t) &
        (df_ariza['OLAYTARIHI'] <= kaza_t + timedelta(days=30))
    ).sum()
    post_kaza_ariza.append(n_ariza)

post_kaza_toplam = sum(post_kaza_ariza)
post_kaza_aracgun = len(df_kaza) * 30  # her kaza × 30 gun
post_kaza_orani = post_kaza_toplam / post_kaza_aracgun

# Rate ratio = post / baseline
rate_ratio = post_kaza_orani / baseline_orani

# Poisson exact CI (rate ratio için)
# Standard Poisson CI: lambda ± z*sqrt(lambda/n)
# Rate ratio CI: log-transform
import math
se_log_rr = math.sqrt(1/post_kaza_toplam + 1/len(baseline_ariza))
log_rr = math.log(rate_ratio)
ci_low  = math.exp(log_rr - 1.96 * se_log_rr)
ci_high = math.exp(log_rr + 1.96 * se_log_rr)

# Poisson hipotez testi (rate eşit mi?)
# z = (rate_post - rate_base) / sqrt(rate_base * (1/n_post + 1/n_base))
# Veya basitçe Poisson test (tek tarafli)
# H0: post_orani = baseline_orani
# H1: post_orani > baseline_orani
# Z-test (large sample Poisson)
ortak_orani = (post_kaza_toplam + len(baseline_ariza)) / (post_kaza_aracgun + baseline_aracgun)
se_diff = math.sqrt(ortak_orani * (1/post_kaza_aracgun + 1/baseline_aracgun))
z_stat = (post_kaza_orani - baseline_orani) / se_diff
from scipy.stats import norm
p_value = 1 - norm.cdf(z_stat)  # tek tarafli (post > base)

print('=== KAZA SONRASI FIRTINA TESTI (Poisson Rate Ratio) ===')
print(f'Baseline orani:        {baseline_orani:.5f} ariza/arac/gun')
print(f'Post-kaza orani:       {post_kaza_orani:.5f} ariza/arac/gun')
print(f'Rate Ratio:            {rate_ratio:.3f}')
print(f'%95 CI:                [{ci_low:.3f}, {ci_high:.3f}]')
print(f'Z-statistic:           {z_stat:.2f}')
print(f'p-value (tek tarafli): {p_value:.6f}')
print()

# Yorum (veriden)
if p_value < 0.05:
    if rate_ratio > 1:
        kat = rate_ratio
        print(f'VERIYE GORE: Kaza sonrasi 30 gunde ariza orani baseline\'in {kat:.2f} KATIDIR (p={p_value:.4f}).')
        print(f'  Kaza sonrasi gizli hasar -> ariza firtinasi VERIYLE TEYIT EDILDI.')
    else:
        print(f'VERIYE GORE: Kaza sonrasi 30 gunde ariza orani baseline\'in {rate_ratio:.2f} KATIDIR (anlamli ama dusuk).')
else:
    print(f'VERIYE GORE: Kaza sonrasi ariza orani baseline\'dan istatistiksel olarak FARKSIZ (p={p_value:.4f}).')
    print('  "Kaza sonrasi firtina" hipotezi REDDEDILEMEZ (yeterince guclu kanit yok).')


=== KAZA SONRASI FIRTINA TESTI (Poisson Rate Ratio) ===
Baseline orani:        0.09387 ariza/arac/gun
Post-kaza orani:       0.08926 ariza/arac/gun
Rate Ratio:            0.951
%95 CI:                [0.924, 0.978]
Z-statistic:           -3.49
p-value (tek tarafli): 0.999759

VERIYE GORE: Kaza sonrasi ariza orani baseline'dan istatistiksel olarak FARKSIZ (p=0.9998).
  "Kaza sonrasi firtina" hipotezi REDDEDILEMEZ (yeterince guclu kanit yok).


---

## 4. Sistem Bazlı Hassasiyet (Chi-square + Bonferroni)
Hangi arıza kategorileri kazadan istatistiksel olarak ANLAMLI etkileniyor?
Çoklu test düzeltmesi: Bonferroni.

In [5]:
# BOLUM 4: Sistem Bazli Hassasiyet (Chi-square)
# Soru: Hangi ariza kategorisi kazadan ANLAMLI olarak etkileniyor?
# Yontem: Her kategori icin (kaza-yakin vs baseline) chi-square + Bonferroni duzeltmesi

# Kategori × (kaza_yakin) tablosu
kat_dist = df_ariza_idx.groupby(['ARIZAUSTKODTANIM', 'kaza_yakin']).size().unstack(fill_value=0)
kat_dist.columns = ['baseline', 'kaza_yakin']

# Min 50 baseline kayit filtresi (gurultu eleme)
kat_dist = kat_dist[kat_dist['baseline'] >= 50]

# Her kategori icin chi-square
sonuclar = []
for kat in kat_dist.index:
    a = kat_dist.loc[kat, 'kaza_yakin']
    b = kat_dist.loc[kat, 'baseline']
    c = kat_dist['kaza_yakin'].sum() - a
    d = kat_dist['baseline'].sum() - b

    table = [[a, b], [c, d]]
    try:
        chi2, p, _, _ = stats.chi2_contingency(table, correction=False)
        # Etki yonu: orani kıyaslarsak
        oran_kaza_yakin = a / (a + b) if (a+b) > 0 else 0
        oran_diger      = c / (c + d) if (c+d) > 0 else 0
        oran_farki = oran_kaza_yakin - oran_diger
        sonuclar.append({
            'Kategori': kat,
            'baseline': b,
            'kaza_yakin': a,
            'oran_kaza_yakin': oran_kaza_yakin,
            'oran_diger':      oran_diger,
            'oran_farki':      oran_farki,
            'chi2': chi2,
            'p': p
        })
    except Exception:
        pass

chi_df = pd.DataFrame(sonuclar)
# Bonferroni correction
n_test = len(chi_df)
chi_df['p_bonferroni'] = (chi_df['p'] * n_test).clip(upper=1.0)
chi_df['anlamli_bonf'] = chi_df['p_bonferroni'] < 0.05

# Sirala: en yuksek pozitif oran farki (kaza sonrasi en cok artan)
chi_df = chi_df.sort_values('oran_farki', ascending=False)

print('=== SISTEM BAZLI HASSASIYET (Chi-square + Bonferroni) ===')
print(f'Test edilen kategori: {n_test} (min 50 baseline kayit)')
print()
print('--- KAZA-YAKIN ARTAN KATEGORILER (TOP 10) ---')
print(f'{"Kategori":40s}  base  yakin  oran_fark   chi2     p_bonf  anlamli')
for _, r in chi_df.head(10).iterrows():
    print(f'{r.Kategori[:40]:40s}  {r.baseline:5}  {r.kaza_yakin:5}  {r.oran_farki:+.4f}    {r.chi2:6.1f}  {r.p_bonferroni:.4f}  {"EVET" if r.anlamli_bonf else "hayir"}')

print()
print('--- KAZA-YAKIN AZALAN (TOP 5, ters yon) ---')
for _, r in chi_df.tail(5).iterrows():
    print(f'{r.Kategori[:40]:40s}  {r.baseline:5}  {r.kaza_yakin:5}  {r.oran_farki:+.4f}    {r.chi2:6.1f}  {r.p_bonferroni:.4f}  {"EVET" if r.anlamli_bonf else "hayir"}')

print()
anlamli_artan = chi_df[(chi_df['oran_farki'] > 0) & (chi_df['anlamli_bonf'])]
anlamli_azalan = chi_df[(chi_df['oran_farki'] < 0) & (chi_df['anlamli_bonf'])]
print(f'VERIYE GORE: Kaza-yakin pencerede ANLAMLI ARTAN: {len(anlamli_artan)} kategori')
print(f'              Kaza-yakin pencerede ANLAMLI AZALAN: {len(anlamli_azalan)} kategori')
if len(anlamli_artan) > 0:
    print(f'  Artan ornek: {", ".join(anlamli_artan.head(3)["Kategori"].tolist())}')


=== SISTEM BAZLI HASSASIYET (Chi-square + Bonferroni) ===
Test edilen kategori: 27 (min 50 baseline kayit)

--- KAZA-YAKIN ARTAN KATEGORILER (TOP 10) ---
Kategori                                  base  yakin  oran_fark   chi2     p_bonf  anlamli
KAROSER ARIZALARI                          3440    759  +0.0994     477.7  0.0000  EVET
AKBİL ARIZALARI                            1780    232  +0.0278      18.6  0.0004  EVET
KAMERA SİSTEMİ ARIZALARI                   1358    153  +0.0131       3.1  1.0000  hayir
YAKIT ve ENJEKSİYON ARIZALARI              1572    177  +0.0131       3.6  1.0000  hayir
SÜSPANSİYON SİSTEMİ ARIZALARI              3054    333  +0.0104       4.3  1.0000  hayir
ROT AYARLARI                                350     37  +0.0072       0.2  1.0000  hayir
BASINÇLI HAVA HATTI ARIZASI                 435     43  +0.0015       0.0  1.0000  hayir
LASTİK ARIZALARI                            462     45  +0.0003       0.0  1.0000  hayir
MOTOR ARIZALARI                            5

---

## 5. Etki Süresi Eğrisi (Gün-gün Wave)
Kaza etkisi ne kadar sürede sönüyor? -7 ile +60 gün arası eğri.

In [6]:
# BOLUM 5: Etki Suresi Egrisi (Gun-gun Wave)
# Soru: Kaza sonrasi etki ne kadar surede sonuyor?
# Yontem: -7 ila +60 gun arasi gun-gun ariza yogunlugu

# Tum kaza-ariza ciftleri icin gun farki
ariza_kaza_full = df_ariza.merge(
    df_kaza[['kapino', 'kazasaat']], left_on='KAPINO', right_on='kapino', how='inner'
)
ariza_kaza_full['gun_farki'] = (
    ariza_kaza_full['OLAYTARIHI'] - ariza_kaza_full['kazasaat']
).dt.total_seconds() / 86400
ariza_kaza_full['gun_farki_int'] = ariza_kaza_full['gun_farki'].round().astype(int)

# -7..+60 gun penceresi
window_df = ariza_kaza_full[(ariza_kaza_full['gun_farki_int'] >= -7) & (ariza_kaza_full['gun_farki_int'] <= 60)]
gun_dist = window_df.groupby('gun_farki_int').size().reindex(range(-7, 61), fill_value=0)

# Her gun icin: araba × pencere içinde olan kaza sayısı
# Her kaza için -7..+60 = 68 gunluk pencere acar
n_kaza = len(df_kaza)
expected_aracgun_per_day = n_kaza  # her gun icin n_kaza pencere icinde

# Gunluk ariza orani
gun_orani = gun_dist / expected_aracgun_per_day

print('=== GUN-GUN ETKI EGRISI ===')
print('Gun       Ariza   Orani   Baseline_Carpani')
for gun in [-7, -3, -1, 0, 1, 3, 7, 14, 30, 45, 60]:
    if gun in gun_orani.index:
        n = gun_dist.loc[gun]
        oran = gun_orani.loc[gun]
        carpan = oran / baseline_orani if baseline_orani > 0 else 0
        print(f'{gun:+3d}.gun  {n:5d}   {oran:.5f}   x{carpan:.2f}')

# Etki ne zaman sonuyor? (baseline'in 1.5 katinin altina dustu mu?)
post_window = gun_orani.loc[1:]
threshold = baseline_orani * 1.5
under_threshold = post_window[post_window < threshold]
if len(under_threshold) > 0:
    iyilesme_gun = under_threshold.index[0]
    print()
    print(f'VERIYE GORE: Etki ~{iyilesme_gun}. gunde baseline*1.5 altina iniyor.')
else:
    print()
    print('VERIYE GORE: 60 gun boyunca etki baseline*1.5 ustunde kaliyor — uzun sureli hasar.')

# Gorsel
fig = px.line(
    x=gun_orani.index, y=gun_orani.values,
    labels={'x': 'Gun farki (kaza=0)', 'y': 'Ariza orani (ariza/arac/gun)'},
    title='Kaza Etki Egrisi: Gun-gun ariza yogunlugu'
)
fig.add_hline(y=baseline_orani, line_dash='dash', line_color='gray',
              annotation_text=f'Baseline ({baseline_orani:.5f})')
fig.add_vline(x=0, line_color='red', annotation_text='Kaza Anı')
fig.show()


=== GUN-GUN ETKI EGRISI ===
Gun       Ariza   Orani   Baseline_Carpani
 -7.gun    169   0.08592   x0.92
 -3.gun    179   0.09100   x0.97
 -1.gun    177   0.08998   x0.96
 +0.gun    707   0.35943   x3.83
 +1.gun    154   0.07829   x0.83
 +3.gun    172   0.08744   x0.93
 +7.gun    183   0.09304   x0.99
+14.gun    185   0.09405   x1.00
+30.gun    156   0.07931   x0.84
+45.gun    124   0.06304   x0.67
+60.gun     94   0.04779   x0.51

VERIYE GORE: Etki ~1. gunde baseline*1.5 altina iniyor.


---

## 6. Kaza Şiddeti × Etki (ANOVA)
Kaza kategorisine göre sonraki arıza ciddiyeti farklı mı?

In [7]:
# BOLUM 6: Kaza Siddeti × Etki (ANOVA)
# Soru: Kaza siddetine (kategori) gore sonraki ariza ciddiyeti farkli mi?
# Yontem: ANOVA — kaza kategorileri arasinda 30g sonra ortalama ciddiyet skoru

# Kaza kategorilerini incele
print('=== KAZA KATEGORI DAGILIMI ===')
print(df_kaza['kategori'].value_counts())
print()

# Her kazanin 30g sonra ortalama ciddiyet skoru
kaza_post_skor = []
for _, row in df_kaza.iterrows():
    kapino = row['kapino']
    kaza_t = row['kazasaat']
    kategori = row['kategori']
    sub = df_ariza[
        (df_ariza['KAPINO'] == kapino) &
        (df_ariza['OLAYTARIHI'] > kaza_t) &
        (df_ariza['OLAYTARIHI'] <= kaza_t + timedelta(days=30))
    ]
    if len(sub) > 0:
        kaza_post_skor.append({
            'kategori': kategori,
            'ort_ciddiyet': sub['ciddiyet_skoru'].mean(),
            'n_ariza': len(sub),
            'max_ciddiyet': sub['ciddiyet_skoru'].max()
        })

kaza_skor_df = pd.DataFrame(kaza_post_skor)

print('=== KAZA SIDDETI × POST-30G ARIZA CIDDIYETI ===')
print(f'{"Kategori":20s}  n_kaza  ort_ariza_n   ort_skor   med_skor   max_skor')
for kat, g in kaza_skor_df.groupby('kategori'):
    if len(g) < 5:
        continue
    print(f'{str(kat)[:20]:20s}  {len(g):6d}  {g["n_ariza"].mean():10.1f}    {g["ort_ciddiyet"].mean():.3f}    {g["ort_ciddiyet"].median():.3f}    {g["max_ciddiyet"].mean():.3f}')

# ANOVA: kategoriler arasinda anlamli fark var mi?
gruplar = [g['ort_ciddiyet'].values for _, g in kaza_skor_df.groupby('kategori') if len(g) >= 5]
if len(gruplar) >= 2:
    f_stat, p_anova = stats.f_oneway(*gruplar)
    print()
    print(f'ANOVA F-stat: {f_stat:.3f}  p-value: {p_anova:.4f}')
    if p_anova < 0.05:
        print('VERIYE GORE: Kaza kategorileri arasi post-ariza ciddiyet farki ANLAMLI (p<0.05).')
    else:
        print('VERIYE GORE: Kaza kategorileri arasi anlamli fark YOK (p>=0.05).')
else:
    print('Yeterli grup yok ANOVA icin.')

# Korelasyon: kaza kategori (ordinal varsayalim) × ariza ciddiyet
print()
print('=== KATEGORI BAZLI POST-ARIZA SAYISI ===')
n_per_kat = kaza_skor_df.groupby('kategori')['n_ariza'].agg(['mean','median','std','count']).round(2)
print(n_per_kat.to_string())


=== KAZA KATEGORI DAGILIMI ===
kategori
Araca Çarpma              1594
Biz bize                   195
Tek Taraflı(Metrobüs)       66
Yayaya Çarpma               37
Biz Bize (Metrobüs)         31
Diğer araçlar               30
Sivil Araç(Metrobüs)        12
Görevli Araç(Metrobüs)       2
Name: count, dtype: int64

=== KAZA SIDDETI × POST-30G ARIZA CIDDIYETI ===
Kategori              n_kaza  ort_ariza_n   ort_skor   med_skor   max_skor
Araca Çarpma            1349         3.1    3.399    3.290    4.704
Biz Bize (Metrobüs)       29         4.8    3.530    3.280    4.957
Biz bize                 168         2.9    3.517    3.259    4.802
Diğer araçlar             26         3.0    3.677    3.720    4.728
Sivil Araç(Metrobüs)      10         3.6    3.556    3.283    4.380
Tek Taraflı(Metrobüs      57         4.4    3.748    3.583    5.171
Yayaya Çarpma             35         3.2    2.994    2.740    4.157

ANOVA F-stat: 1.414  p-value: 0.2053
VERIYE GORE: Kaza kategorileri arasi anlamli far

---

## 7. Kaza Öncesi Uyarı (Pre-Kaza)
48 saat / 7 gün / 30 gün pre-kaza arıza oranları, baseline ile karşılaştırma.

In [8]:
# BOLUM 7: Kaza Oncesi Uyari (Pre-Kaza Ariza Orani)
# Soru: Kazadan onceki 48s/7g/30g donemde ariza orani baseline'a gore yuksek mi?
# Yontem: 3 farkli pencere icin Poisson rate ratio

pencereler = [(2, '48 saat'), (7, '7 gun'), (30, '30 gun')]
print('=== KAZA ONCESI UYARI TESTLERI ===')
print(f'{"Pencere":12s}  ariza_n  arac_gun_n  orani       rate_ratio  p_value')

for gun, ad in pencereler:
    pre_ariza_n = 0
    for _, row in df_kaza.iterrows():
        kapino = row['kapino']
        kaza_t = row['kazasaat']
        n = (
            (df_ariza['KAPINO'] == kapino) &
            (df_ariza['OLAYTARIHI'] >= kaza_t - timedelta(days=gun)) &
            (df_ariza['OLAYTARIHI'] < kaza_t)
        ).sum()
        pre_ariza_n += n

    pre_aracgun = len(df_kaza) * gun
    pre_orani = pre_ariza_n / pre_aracgun
    rr = pre_orani / baseline_orani

    # Poisson z-test
    ortak = (pre_ariza_n + len(baseline_ariza)) / (pre_aracgun + baseline_aracgun)
    se = math.sqrt(ortak * (1/pre_aracgun + 1/baseline_aracgun))
    z = (pre_orani - baseline_orani) / se if se > 0 else 0
    p = 1 - norm.cdf(z) if z > 0 else norm.cdf(z) * 2

    print(f'{ad:12s}  {pre_ariza_n:7d}  {pre_aracgun:10d}  {pre_orani:.5f}     x{rr:.3f}    {p:.6f}')

print()
# Bu pencerelerden hangisi en guclu uyari?
# Veriyi yorumla


=== KAZA ONCESI UYARI TESTLERI ===
Pencere       ariza_n  arac_gun_n  orani       rate_ratio  p_value
48 saat           433        3934  0.11007     x1.173    0.000480
7 gun            1275       13769  0.09260     x0.986    0.630372
30 gun           4656       59010  0.07890     x0.841    0.000000



---

## 8. Güvenlik Kritik Lift Skoru
Hangi kategori kazaya en yakın ortaya çıkar (kaza habercisi)?

In [9]:
# BOLUM 8: Guvenlik Kritik Lift Skoru
# Soru: Hangi kategorideki ariza kazaya en yakin ortaya cikar (kaza habercisi)?
# Lift = P(ariza | kazadan_onceki_48s) / P(ariza | herhangi_zaman)

# Kazadan onceki 48 saatte goruluen arizalar
pre_48s_set = []
for _, row in df_kaza.iterrows():
    kapino = row['kapino']
    kaza_t = row['kazasaat']
    sub = df_ariza[
        (df_ariza['KAPINO'] == kapino) &
        (df_ariza['OLAYTARIHI'] >= kaza_t - timedelta(hours=48)) &
        (df_ariza['OLAYTARIHI'] < kaza_t)
    ]
    pre_48s_set.append(sub)

pre_48s_df = pd.concat(pre_48s_set) if pre_48s_set else pd.DataFrame()
print(f'Kaza oncesi 48s ariza sayisi: {len(pre_48s_df)}')

# Lift hesabi her kategori icin
print()
print('=== LIFT SKORU (Kaza Habercisi Sıralaması) ===')
print('Lift > 1 = kategori kazaya yakin daha sik gozukur (haberci)')
print('Lift < 1 = kategori kazaya yakin daha az gozukur')
print()

global_kat_orani = df_ariza['ARIZAUSTKODTANIM'].value_counts(normalize=True)
pre_kat_orani = pre_48s_df['ARIZAUSTKODTANIM'].value_counts(normalize=True) if len(pre_48s_df) > 0 else pd.Series()

lift_sonuc = []
for kat in global_kat_orani.index:
    if kat in pre_kat_orani.index:
        lift = pre_kat_orani[kat] / global_kat_orani[kat]
    else:
        lift = 0
    n_pre = (pre_48s_df['ARIZAUSTKODTANIM'] == kat).sum() if len(pre_48s_df) > 0 else 0
    if n_pre >= 5:  # Min eşik gurultu için
        lift_sonuc.append({
            'Kategori': kat,
            'global_orani': global_kat_orani[kat],
            'pre_kaza_orani': pre_kat_orani.get(kat, 0),
            'lift': lift,
            'n_pre_kaza': n_pre
        })

lift_df = pd.DataFrame(lift_sonuc).sort_values('lift', ascending=False)

print(f'{"Kategori":40s}  n_pre  global%  pre_kaza%  Lift')
for _, r in lift_df.head(15).iterrows():
    print(f'{r.Kategori[:40]:40s}  {r.n_pre_kaza:5d}  {r.global_orani*100:6.2f}  {r.pre_kaza_orani*100:8.2f}  {r.lift:.2f}')

# Guvenlik kritik filtresi
guvenlik_kategoriler = ['FREN ŞİKAYETLERİ', 'DİREKSİYON ARIZALARI', 'LASTİK ARIZALARI', 'KAPI ARIZALARI']
guvenlik_lift = lift_df[lift_df['Kategori'].isin(guvenlik_kategoriler)]
print()
print('=== GUVENLIK KRITIK KATEGORILER ===')
if len(guvenlik_lift) > 0:
    print(guvenlik_lift.to_string(index=False))
    ort_guvenlik_lift = guvenlik_lift['lift'].mean()
    print(f'Guvenlik kritik ortalama lift: {ort_guvenlik_lift:.2f}')
else:
    print('Guvenlik kategorilerinde yeterli pre-kaza kayit yok.')


Kaza oncesi 48s ariza sayisi: 433

=== LIFT SKORU (Kaza Habercisi Sıralaması) ===
Lift > 1 = kategori kazaya yakin daha sik gozukur (haberci)
Lift < 1 = kategori kazaya yakin daha az gozukur

Kategori                                  n_pre  global%  pre_kaza%  Lift
KAROSER ARIZALARI                           107    7.17     24.71  3.45
LASTİK ARIZALARI                              6    0.87      1.39  1.60
AKBİL ARIZALARI                              23    3.44      5.31  1.55
YAKIT ve ENJEKSİYON ARIZALARI                19    2.99      4.39  1.47
BASINÇLI HAVA HATTI ARIZASI                   5    0.82      1.15  1.41
KAYIŞ KASNAK ARIZALARI                        8    1.35      1.85  1.37
SOĞUTMA SİSTEMİ ARIZASI                      49   10.90     11.32  1.04
BASINÇLI HAVA DONANIMI ARIZALARI              7    1.68      1.62  0.96
FREN ŞİKAYETLERİ                             26    6.57      6.00  0.91
Destek                                       14    3.56      3.23  0.91
MOTOR ARIZALAR

---

## 9. Tekrar Kaza Patterni
Tekrar kaza yapan araçlar daha yaşlı/sorunlu mu?

In [10]:
# BOLUM 9: Tekrar Kaza Patterni
# Soru: Tekrar kaza yapan araclar daha yasli/sorunlu mu?
# Yontem: 1, 2, 3+ kaza yapan araclarin yas + ariza profil karsilastirmasi

# Arac basina kaza sayisi
arac_kaza_n = df_kaza.groupby('kapino').size().reset_index(name='kaza_sayisi')

# Arac yaslarini ekle (mode)
arac_yas_x = df_ariza.groupby('KAPINO')['MODELYILI'].agg(lambda x: x.mode().iloc[0]).reset_index()
arac_yas_x['arac_yasi'] = 2025 - arac_yas_x['MODELYILI']
arac_yas_x = arac_yas_x.rename(columns={'KAPINO': 'kapino'})

arac_kaza_yas = arac_kaza_n.merge(arac_yas_x[['kapino', 'arac_yasi']], on='kapino', how='left')
arac_kaza_yas = arac_kaza_yas.dropna()

print('=== TEKRAR KAZA × YAS ===')
print(f'{"Kaza Sayisi":15s}  n_arac  ort_yas   med_yas')
for n in [1, 2, 3, 4, 5]:
    if n < 5:
        sub = arac_kaza_yas[arac_kaza_yas['kaza_sayisi'] == n]
        etiket = f'{n} kaza'
    else:
        sub = arac_kaza_yas[arac_kaza_yas['kaza_sayisi'] >= 5]
        etiket = '5+ kaza'
    if len(sub) >= 5:
        print(f'{etiket:15s}  {len(sub):6d}  {sub["arac_yasi"].mean():5.1f}    {sub["arac_yasi"].median():5.1f}')

# Tekrar kaza yapan araclar genel filodan farklı yas grubunda mı?
tek_kaza   = arac_kaza_yas[arac_kaza_yas['kaza_sayisi'] == 1]['arac_yasi']
cok_kaza   = arac_kaza_yas[arac_kaza_yas['kaza_sayisi'] >= 3]['arac_yasi']

if len(tek_kaza) > 10 and len(cok_kaza) > 10:
    u, p = stats.mannwhitneyu(cok_kaza, tek_kaza, alternative='greater')
    print()
    print(f'=== Mann-Whitney U (cok kaza vs tek kaza yasli mi) ===')
    print(f'Tek kaza ort yas: {tek_kaza.mean():.1f}  (n={len(tek_kaza)})')
    print(f'Cok kaza ort yas: {cok_kaza.mean():.1f}  (n={len(cok_kaza)})')
    print(f'U-stat: {u:.0f}  p-value: {p:.4f}')
    fark = cok_kaza.mean() - tek_kaza.mean()
    if p < 0.05:
        if fark > 0:
            print(f'VERIYE GORE: Cok kaza yapan araclar tek kaza yapanlardan {fark:+.1f} yil DAHA YASLI (p<0.05).')
        else:
            print(f'VERIYE GORE: Cok kaza yapan araclar tek kaza yapanlardan {abs(fark):.1f} yil DAHA GENC (p<0.05).')
    else:
        print(f'VERIYE GORE: Tekrar kaza × yas iliskisi anlamsiz (p={p:.4f}).')

# Korelasyon: kaza_sayisi ile yas
r_yas = arac_kaza_yas['arac_yasi'].corr(arac_kaza_yas['kaza_sayisi'])
print()
print(f'Pearson korelasyon (yas × kaza_sayisi): r = {r_yas:+.3f}')


=== TEKRAR KAZA × YAS ===
Kaza Sayisi      n_arac  ort_yas   med_yas
1 kaza             1011   11.6     12.0
2 kaza              305   11.0     12.0
3 kaza               76   11.9     12.0
4 kaza               22   13.5     12.0
5+ kaza               6   14.3     12.0

=== Mann-Whitney U (cok kaza vs tek kaza yasli mi) ===
Tek kaza ort yas: 11.6  (n=1011)
Cok kaza ort yas: 12.3  (n=104)
U-stat: 56496  p-value: 0.0925
VERIYE GORE: Tekrar kaza × yas iliskisi anlamsiz (p=0.0925).

Pearson korelasyon (yas × kaza_sayisi): r = +0.023


---

## 10. Confounder Kontrolü — Yaş + Yoğunluk
Kaza-yakın fırtına gerçekten kazadan mı, yoksa "kaza yapan araçlar zaten sorunlu" oldukları için mi?

In [11]:
# BOLUM 10: Confounder — Yas + Sefer Yogunlugu
# Soru: Kaza-yakin firtina gercekten kazadan mi, yoksa "kaza yapan araclar zaten sorunlu" oldugu icin mi?
# Yontem: Kaza yapan araclar vs genel filo yas/sefer karsilastirmasi

# Kaza yapan araclarin profili
kaza_arac_set = set(df_kaza['kapino'])
df_ariza_idx['kaza_yapti'] = df_ariza_idx['KAPINO'].isin(kaza_arac_set)

# Yas dağılımı karsilastirmasi
arac_profil = df_ariza_idx.groupby('KAPINO').agg(
    yas=('MODELYILI', lambda x: 2025 - x.mode().iloc[0]),
    kaza_yapti=('kaza_yapti', 'first'),
    toplam_ariza=('KAPINO', 'count'),
    ort_ciddiyet=('ciddiyet_skoru', 'mean')
).reset_index()

kaza_yapan = arac_profil[arac_profil['kaza_yapti']]
kaza_yapmayan = arac_profil[~arac_profil['kaza_yapti']]

print('=== CONFOUNDER 1: YAS DAGILIMI ===')
print(f'{"Grup":20s}  n_arac  ort_yas  med_yas')
print(f'{"Kaza yapan":20s}  {len(kaza_yapan):6d}  {kaza_yapan["yas"].mean():5.1f}    {kaza_yapan["yas"].median():.1f}')
print(f'{"Kaza yapmayan":20s}  {len(kaza_yapmayan):6d}  {kaza_yapmayan["yas"].mean():5.1f}    {kaza_yapmayan["yas"].median():.1f}')

if len(kaza_yapan) > 10 and len(kaza_yapmayan) > 10:
    u, p_mw = stats.mannwhitneyu(kaza_yapan['yas'], kaza_yapmayan['yas'], alternative='two-sided')
    fark_yas = kaza_yapan['yas'].mean() - kaza_yapmayan['yas'].mean()
    print(f'Mann-Whitney U: u={u:.0f}, p={p_mw:.4f}')
    if p_mw < 0.05:
        yon = 'YASLI' if fark_yas > 0 else 'GENC'
        print(f'VERIYE GORE: Kaza yapan araclar daha {yon} (fark: {fark_yas:+.1f} yil, p<0.05).')
        print('  -> Yas dominant confounder olabilir. Kaza-yakin firtinanin bir kismi yas kaynakli.')
    else:
        print(f'VERIYE GORE: Yas dagiliminda anlamli fark yok (p={p_mw:.4f}). Confounder degil.')

# Confounder 2: Toplam ariza sayisi (genel sorunluluk)
print()
print('=== CONFOUNDER 2: GENEL ARIZA YOGUNLUGU ===')
print(f'{"Grup":20s}  ort_ariza  med_ariza  ort_ciddiyet')
print(f'{"Kaza yapan":20s}  {kaza_yapan["toplam_ariza"].mean():9.1f}  {kaza_yapan["toplam_ariza"].median():9.1f}  {kaza_yapan["ort_ciddiyet"].mean():.3f}')
print(f'{"Kaza yapmayan":20s}  {kaza_yapmayan["toplam_ariza"].mean():9.1f}  {kaza_yapmayan["toplam_ariza"].median():9.1f}  {kaza_yapmayan["ort_ciddiyet"].mean():.3f}')

u2, p_ariza = stats.mannwhitneyu(kaza_yapan['toplam_ariza'], kaza_yapmayan['toplam_ariza'], alternative='greater')
print(f'Mann-Whitney U (kaza yapan daha cok ariza): p={p_ariza:.4f}')
if p_ariza < 0.05:
    print('VERIYE GORE: Kaza yapan araclar GENEL OLARAK daha cok ariza yapiyor.')
    print('  -> Kaza-yakin firtinanin bir kismi araclarin "zaten sorunlu" olmasindan olabilir.')

# YAS KONTROL ALTINDA: rate ratio'yu yeniden hesapla
# Strafyle: yas grubuna göre kaza-yakin oranı
print()
print('=== YAS KONTROL ALTINDA RATE RATIO ===')
df_ariza_idx['arac_yas'] = df_ariza_idx['KAPINO'].map(arac_profil.set_index('KAPINO')['yas'])
df_ariza_idx['yas_grup'] = pd.cut(df_ariza_idx['arac_yas'], bins=[0, 8, 13, 25],
                                    labels=['Genc(0-8)', 'Orta(9-13)', 'Yasli(14+)'])

print(f'{"Yas Grubu":15s}  baseline_n  yakin_n  rate_ratio')
for grup in ['Genc(0-8)', 'Orta(9-13)', 'Yasli(14+)']:
    sub = df_ariza_idx[df_ariza_idx['yas_grup'] == grup]
    base_n = (~sub['kaza_yakin']).sum()
    yakin_n = sub['kaza_yakin'].sum()
    if base_n > 100:
        rr_grup = yakin_n / max(base_n, 1)
        print(f'{grup:15s}  {base_n:10d}  {yakin_n:7d}  {rr_grup:.4f}')


=== CONFOUNDER 1: YAS DAGILIMI ===
Grup                  n_arac  ort_yas  med_yas
Kaza yapan              1420   11.5    12.0
Kaza yapmayan           2089   11.6    12.0
Mann-Whitney U: u=1429030, p=0.0559
VERIYE GORE: Yas dagiliminda anlamli fark yok (p=0.0559). Confounder degil.

=== CONFOUNDER 2: GENEL ARIZA YOGUNLUGU ===
Grup                  ort_ariza  med_ariza  ort_ciddiyet
Kaza yapan                 16.0       14.0  3.455
Kaza yapmayan              17.2       16.0  3.631
Mann-Whitney U (kaza yapan daha cok ariza): p=0.9995

=== YAS KONTROL ALTINDA RATE RATIO ===
Yas Grubu        baseline_n  yakin_n  rate_ratio
Genc(0-8)              8237      827  0.1004
Orta(9-13)            34071     3590  0.1054
Yasli(14+)            11074      760  0.0686


---

## 11. ML Feature Türetme + Korelasyon
5 yeni feature kandidatı + ciddiyet_skoru ile korelasyon testi.

In [12]:
# BOLUM 11: ML Feature Turetme + Korelasyon
# Soru: Kaza analizinden ML modeline hangi feature girmeli?
# 5 yeni feature olusturup ciddiyet_skoru ile korelasyonunu test et.

# Her arac icin kaza profili
arac_kaza_features = []
for kapino in df_ariza_idx['KAPINO'].unique():
    arac_kazalari = df_kaza[df_kaza['kapino'] == kapino]
    if len(arac_kazalari) == 0:
        arac_kaza_features.append({
            'KAPINO': kapino,
            'gecmis_kaza_sayisi': 0,
            'son_kaza_gun': 999,  # 'hic kaza yapmadi' icin yuksek deger
            'tekrar_kaza_riski': 0
        })
    else:
        son_kaza = arac_kazalari['kazasaat'].max()
        son_ariza = df_ariza_idx[df_ariza_idx['KAPINO'] == kapino]['OLAYTARIHI'].max()
        arac_kaza_features.append({
            'KAPINO': kapino,
            'gecmis_kaza_sayisi': len(arac_kazalari),
            'son_kaza_gun': max((son_ariza - son_kaza).days, 0),
            'tekrar_kaza_riski': 1 if len(arac_kazalari) >= 2 else 0
        })

kaza_feat_df = pd.DataFrame(arac_kaza_features)

# Arac başına ortalama ciddiyet skoru
arac_skor = df_ariza.groupby('KAPINO')['ciddiyet_skoru'].agg(['mean', 'count']).reset_index()
arac_skor.columns = ['KAPINO', 'ort_skor', 'ariza_n']

# Birlestir
ml_df = arac_skor.merge(kaza_feat_df, on='KAPINO', how='left')
ml_df = ml_df[ml_df['ariza_n'] >= 5]  # gurultu eleme

print(f'Analiz seti: {len(ml_df):,} arac (min 5 ariza)')
print()
print('=== FEATURE TURETME × CIDDIYET_SKORU KORELASYON ===')
print(f'{"Feature":30s}  r        p        n')

features_to_test = ['gecmis_kaza_sayisi', 'son_kaza_gun', 'tekrar_kaza_riski']
for f in features_to_test:
    r, p = stats.pearsonr(ml_df[f], ml_df['ort_skor'])
    n_nonzero = (ml_df[f] != 0).sum() if f != 'son_kaza_gun' else (ml_df[f] != 999).sum()
    print(f'{f:30s}  {r:+.4f}  {p:.4f}  {n_nonzero}')

# Spearman da test et (lineer olmayabilir)
print()
print('=== SPEARMAN (sira korelasyonu) ===')
for f in features_to_test:
    rs, ps = stats.spearmanr(ml_df[f], ml_df['ort_skor'])
    print(f'{f:30s}  rs={rs:+.4f}  p={ps:.4f}')

# Bant analizi: kaza sayisi grupları
print()
print('=== KAZA SAYISI BANTLARINA GORE CIDDIYET SKORU ===')
ml_df['kaza_grup'] = pd.cut(ml_df['gecmis_kaza_sayisi'],
                             bins=[-0.1, 0.5, 1.5, 3.5, 100],
                             labels=['Hic', '1', '2-3', '4+'])
print(ml_df.groupby('kaza_grup')['ort_skor'].agg(['count', 'mean', 'std']).round(3))

# ANOVA: kaza grupları arası anlamli mi?
gruplar = [g['ort_skor'].values for _, g in ml_df.groupby('kaza_grup') if len(g) >= 5]
if len(gruplar) >= 2:
    f, p = stats.f_oneway(*gruplar)
    print()
    print(f'ANOVA F={f:.2f}  p={p:.6f}')
    if p < 0.05:
        print('VERIYE GORE: Kaza sayisi gruplari arasinda ciddiyet skoru farki ANLAMLI.')
    else:
        print('VERIYE GORE: Kaza sayisi gruplari arasinda anlamli fark YOK.')

print()
print('=== ML KANDIDAT KARARI ===')
en_iyi = max(features_to_test, key=lambda f: abs(stats.pearsonr(ml_df[f], ml_df['ort_skor'])[0]))
en_iyi_r = stats.pearsonr(ml_df[en_iyi], ml_df['ort_skor'])[0]
print(f'En guclu feature: {en_iyi} (r={en_iyi_r:+.4f})')
if abs(en_iyi_r) > 0.05:
    print(f'  -> ML modeline eklenebilir.')
else:
    print(f'  -> Tum kaza feature\'lari zayif (|r|<0.05). ML\'e dogrudan eklemek faydasiz.')


Analiz seti: 3,316 arac (min 5 ariza)

=== FEATURE TURETME × CIDDIYET_SKORU KORELASYON ===
Feature                         r        p        n
gecmis_kaza_sayisi              -0.1212  0.0000  1353
son_kaza_gun                    +0.1293  0.0000  1353
tekrar_kaza_riski               -0.0922  0.0000  393

=== SPEARMAN (sira korelasyonu) ===
gecmis_kaza_sayisi              rs=-0.1250  p=0.0000
son_kaza_gun                    rs=+0.1169  p=0.0000
tekrar_kaza_riski               rs=-0.0968  p=0.0000

=== KAZA SAYISI BANTLARINA GORE CIDDIYET SKORU ===
           count   mean    std
kaza_grup                     
Hic         1963  3.663  0.652
1            960  3.514  0.701
2-3          365  3.420  0.691
4+            28  3.437  0.484

ANOVA F=20.29  p=0.000000
VERIYE GORE: Kaza sayisi gruplari arasinda ciddiyet skoru farki ANLAMLI.

=== ML KANDIDAT KARARI ===
En guclu feature: son_kaza_gun (r=+0.1293)
  -> ML modeline eklenebilir.


---

## 12. Kaza Anı (t=0) Sistem Detayı
707 arıza patlamasında hangi sistemler etkileniyor? Lift skorlarıyla.

In [13]:
# BOLUM 12: Kaza Aninda (t=0) Sistem Detayi
# 707 ariza patlamasinin sistem dagilimi nedir?
# Hangi sistemler kazada FIZIKSEL olarak zarar goruyor?

# Kaza günü (t=0) ariza kayitlari
kaza_gunu_ariza = ariza_kaza_full[ariza_kaza_full['gun_farki_int'] == 0]
print(f'Kaza gunu (t=0) ariza sayisi: {len(kaza_gunu_ariza):,}')

# Sistem dagilimi
kg_dist = kaza_gunu_ariza['ARIZAUSTKODTANIM'].value_counts()
kg_oran = kaza_gunu_ariza['ARIZAUSTKODTANIM'].value_counts(normalize=True)

# Genel filo dagilimi (kıyas)
genel_oran = df_ariza['ARIZAUSTKODTANIM'].value_counts(normalize=True)

print()
print('=== KAZA ANI (t=0) SISTEM DETAYI ===')
print(f'{"Sistem":40s}  n_kaza_gun  pct_kaza_gun  pct_genel  Lift')
sonuclar_t0 = []
for kat in kg_dist.index[:15]:
    n = kg_dist[kat]
    p_kg = kg_oran[kat]
    p_genel = genel_oran.get(kat, 0)
    lift = p_kg / p_genel if p_genel > 0 else 0
    sonuclar_t0.append({'kategori': kat, 'n': n, 'lift': lift, 'pct_kg': p_kg, 'pct_genel': p_genel})
    print(f'{str(kat)[:40]:40s}  {n:10d}  {p_kg*100:11.2f}  {p_genel*100:8.2f}   {lift:.2f}')

# Lift'i sırala
t0_df = pd.DataFrame(sonuclar_t0).sort_values('lift', ascending=False)
print()
print('=== KAZA ANI YUKSEK LIFT KATEGORILERI ===')
print('(Lift > 2 = bu kategorinin kaza günü görülmesi normal günden 2 kat daha sık)')
yuksek_lift = t0_df[t0_df['lift'] > 2]
if len(yuksek_lift) > 0:
    for _, r in yuksek_lift.iterrows():
        print(f'  {r.kategori[:40]:40s}: lift={r.lift:.2f}  (n={r.n})')
else:
    print('  Yuksek lift kategori yok.')

# Saat etkisi: Kaza günü arızalar saat dagilimina göre kazaya yakin mi?
kaza_gunu_ariza_full = ariza_kaza_full[ariza_kaza_full['gun_farki_int'] == 0].copy()
kaza_gunu_ariza_full['saat_farki'] = (
    kaza_gunu_ariza_full['OLAYTARIHI'] - kaza_gunu_ariza_full['kazasaat']
).dt.total_seconds() / 3600

print()
print('=== KAZA ANI SAAT FARKI (saat cinsinden) ===')
print(f'  Min:    {kaza_gunu_ariza_full["saat_farki"].min():+.1f} saat (kaza ONCESI)')
print(f'  Max:    {kaza_gunu_ariza_full["saat_farki"].max():+.1f} saat (kaza SONRASI)')
print(f'  Median: {kaza_gunu_ariza_full["saat_farki"].median():+.1f} saat')
print()
# Kac arıza kaza ANINDAN sonra (pozitif saat farki)?
sonra_n = (kaza_gunu_ariza_full['saat_farki'] > 0).sum()
once_n = (kaza_gunu_ariza_full['saat_farki'] < 0).sum()
ayni_saat = (kaza_gunu_ariza_full['saat_farki'].abs() <= 1).sum()
print(f'Kazadan ONCE ariza:    {once_n}')
print(f'Kazadan SONRA ariza:   {sonra_n}')
print(f'Kaza ile ±1 saat ICINDE: {ayni_saat}')


Kaza gunu (t=0) ariza sayisi: 707

=== KAZA ANI (t=0) SISTEM DETAYI ===
Sistem                                    n_kaza_gun  pct_kaza_gun  pct_genel  Lift
KAROSER ARIZALARI                                515        72.84      7.17   10.16
SOĞUTMA SİSTEMİ ARIZASI                           30         4.24     10.90   0.39
MOTOR ARIZALARI                                   21         2.97      9.63   0.31
OTOMATİK ŞANZIMAN ARIZALARI                       15         2.12      5.51   0.39
KAPI ARIZALARI                                    15         2.12     10.49   0.20
FREN ŞİKAYETLERİ                                  14         1.98      6.57   0.30
AKBİL ARIZALARI                                   13         1.84      3.44   0.54
ELEKTRİK SİSTEMİ ARIZALARI                        13         1.84     10.14   0.18
KLİMA SİSTEMİ ARIZALARI                           12         1.70      6.73   0.25
Destek                                            11         1.56      3.56   0.44
YAKIT ve ENJE

---

## 13. Vaka Doğrulama — Spesifik Araçlar
Kaza yapan araçlar gerçekten 30 gün içinde arıza yapıyor mu? Spesifik vakalar üzerinden kontrol.

In [14]:
# BOLUM 13: Vaka Dogrulama — Spesifik Kaza Yapan Araclarin Sonrasi
# Soru: Kaza yapan araclar gercekten 30 gun icinde ariza yapiyor mu?
# Yontem: En cok kaza yapan 10 araci sec, bunların pre/post timeline'ini incele.

# En cok kaza yapan 10 arac
en_kaza_arac = df_kaza['kapino'].value_counts().head(10)

print('=== EN COK KAZA YAPAN 10 ARAC — VAKA ANALIZI ===')
print(f'{"Arac":10s}  kaza_n  son_30g_post  ort_arac_yas')

vaka_sonuc = []
for kapino, n_kaza in en_kaza_arac.items():
    kaza_tarihleri = df_kaza[df_kaza['kapino'] == kapino]['kazasaat'].sort_values()

    # Her kazasının sonrasındaki 30 gün arıza
    post_ariza_n_list = []
    for kt in kaza_tarihleri:
        n = (
            (df_ariza['KAPINO'] == kapino) &
            (df_ariza['OLAYTARIHI'] > kt) &
            (df_ariza['OLAYTARIHI'] <= kt + timedelta(days=30))
        ).sum()
        post_ariza_n_list.append(n)

    post_total = sum(post_ariza_n_list)
    yas_row = arac_profil[arac_profil['KAPINO'] == kapino]
    yas = yas_row['yas'].values[0] if len(yas_row) > 0 else None

    vaka_sonuc.append({
        'kapino': kapino,
        'n_kaza': n_kaza,
        'post_total': post_total,
        'yas': yas,
        'kaza_tarihleri': kaza_tarihleri.tolist()
    })
    print(f'{kapino:10s}  {n_kaza:6d}  {post_total:12d}  {yas if yas else "N/A"}')

# Bu 10 arac icin: kazadan sonraki ilk arıza ne kadar sonra?
print()
print('=== KAZADAN SONRAKI ILK ARIZA SURESI (orneklemeler) ===')
ornek_araclar = [v['kapino'] for v in vaka_sonuc[:5]]
for kapino in ornek_araclar:
    kazalar = df_kaza[df_kaza['kapino'] == kapino]['kazasaat'].sort_values()
    arizalar = df_ariza[df_ariza['KAPINO'] == kapino].sort_values('OLAYTARIHI')

    print(f'\n--- {kapino} ---')
    for _, kt in kazalar.head(3).items():
        # Kazadan sonraki ilk arıza
        sonraki = arizalar[arizalar['OLAYTARIHI'] > kt]
        if len(sonraki) > 0:
            ilk = sonraki.iloc[0]
            saat_farki = (ilk['OLAYTARIHI'] - kt).total_seconds() / 3600
            print(f'  Kaza: {kt.strftime("%Y-%m-%d %H:%M")}  -> Ilk ariza {saat_farki:.1f} saat sonra ({ilk["ARIZAUSTKODTANIM"]})')
        else:
            print(f'  Kaza: {kt.strftime("%Y-%m-%d %H:%M")}  -> 30 gun icinde HIC ARIZA YOK')

# Kaza yapan tum araclarin "ilk ariza zamanı" dagilimi
print()
print('=== KAZA SONRASI ILK ARIZA SURESI DAGILIMI (tum kazalar) ===')
ilk_ariza_saatler = []
for _, row in df_kaza.iterrows():
    kapino = row['kapino']
    kt = row['kazasaat']
    sonraki = df_ariza[
        (df_ariza['KAPINO'] == kapino) &
        (df_ariza['OLAYTARIHI'] > kt) &
        (df_ariza['OLAYTARIHI'] <= kt + timedelta(days=30))
    ]
    if len(sonraki) > 0:
        ilk_ariza = sonraki['OLAYTARIHI'].min()
        saat = (ilk_ariza - kt).total_seconds() / 3600
        ilk_ariza_saatler.append(saat)

ilk_ariza_arr = np.array(ilk_ariza_saatler)
n_var = len(ilk_ariza_arr)
n_yok = len(df_kaza) - n_var
print(f'30 gun icinde ariza yapan kaza: {n_var:,} ({n_var/len(df_kaza)*100:.1f}%)')
print(f'30 gun icinde ariza YAPMAYAN: {n_yok:,} ({n_yok/len(df_kaza)*100:.1f}%)')
print(f'Ortalama ilk ariza zamani: {ilk_ariza_arr.mean():.1f} saat ({ilk_ariza_arr.mean()/24:.1f} gun)')
print(f'Median: {np.median(ilk_ariza_arr):.1f} saat ({np.median(ilk_ariza_arr)/24:.1f} gun)')
print()
print(f'Ilk 1 saat icinde:    {(ilk_ariza_arr <= 1).sum()} ({(ilk_ariza_arr <= 1).sum()/n_var*100:.1f}%)')
print(f'Ilk 24 saat icinde:   {(ilk_ariza_arr <= 24).sum()} ({(ilk_ariza_arr <= 24).sum()/n_var*100:.1f}%)')
print(f'Ilk 7 gun icinde:     {(ilk_ariza_arr <= 168).sum()} ({(ilk_ariza_arr <= 168).sum()/n_var*100:.1f}%)')


=== EN COK KAZA YAPAN 10 ARAC — VAKA ANALIZI ===
Arac        kaza_n  son_30g_post  ort_arac_yas
K2418            5            35  12.0
M2194            5            13  19.0
O3517            5            25  12.0
O3346            5            22  12.0
K2791            5            13  12.0
M5619            5             9  19.0
K2237            4            10  12.0
M6397            4             9  13.0
O6735            4             6  12.0
O6974            4            15  12.0

=== KAZADAN SONRAKI ILK ARIZA SURESI (orneklemeler) ===

--- K2418 ---
  Kaza: 2025-01-30 18:06  -> Ilk ariza 0.1 saat sonra (KAROSER ARIZALARI)
  Kaza: 2025-03-11 19:44  -> Ilk ariza 205.3 saat sonra (KAMERA SİSTEMİ ARIZALARI)
  Kaza: 2025-04-14 19:33  -> Ilk ariza 96.0 saat sonra (ISITMA SİSTEMİ)

--- M2194 ---
  Kaza: 2025-01-10 10:40  -> Ilk ariza 146.8 saat sonra (ISITMA SİSTEMİ)
  Kaza: 2025-01-10 10:40  -> Ilk ariza 146.8 saat sonra (ISITMA SİSTEMİ)
  Kaza: 2025-01-10 10:40  -> Ilk ariza 146.8 saat so

---

## 14. Methodolojik Düzeltme — Apples-to-Apples Karşılaştırma
Bölüm 3'teki RR=0.61 sonucu **biased**: TÜM araç baseline vs SADECE kaza yapan post-kaza.
Doğru karşılaştırma: SADECE kaza yapan araçların kaza-olmayan dönemleri ile karşılaştırma.

In [15]:
# BOLUM 14: Methodolojik Duzeltme — Apples-to-Apples Karsilastirma
# Onceki test biased: TÜM araç baseline vs SADECE kaza yapan post-kaza
# Dogru karsilastirma: SADECE kaza yapan araclarin kaza-olmayan donemleri vs kaza sonrasi

# Sadece kaza yapan araclari al
kaza_yapan_kapinolar = set(df_kaza['kapino'].unique())
df_ariza_kazaref = df_ariza_idx[df_ariza_idx['KAPINO'].isin(kaza_yapan_kapinolar)].copy()

# Bu araclarin baseline ariza orani (kaza-yakin olmayan donemler)
baseline_kazaref = df_ariza_kazaref[~df_ariza_kazaref['kaza_yakin']]
baseline_kazaref_aracgun = len(kaza_yapan_kapinolar) * 180 - len(df_kaza) * 32
baseline_kazaref_orani = len(baseline_kazaref) / max(baseline_kazaref_aracgun, 1)

# Post-kaza orani (zaten hesaplı)
# post_kaza_orani

# Yeni rate ratio
rr_dogru = post_kaza_orani / baseline_kazaref_orani if baseline_kazaref_orani > 0 else 0

print('=== METHODOLOJIK DUZELTME ===')
print(f'Kaza yapan arac sayisi: {len(kaza_yapan_kapinolar):,}')
print()
print('--- BASELINE: Sadece kaza yapan araclarin kaza-olmayan donemleri ---')
print(f'Ariza sayisi: {len(baseline_kazaref):,}')
print(f'Arac×gun:     {baseline_kazaref_aracgun:,}')
print(f'Orani:        {baseline_kazaref_orani:.5f} ariza/arac/gun')
print()
print('--- POST-KAZA: 30 gun sonra ---')
print(f'Ariza sayisi: {post_kaza_toplam:,}')
print(f'Arac×gun:     {post_kaza_aracgun:,}')
print(f'Orani:        {post_kaza_orani:.5f} ariza/arac/gun')
print()
print('=== ESKI vs DUZELTILMIS RATE RATIO ===')
print(f'Eski (biased):       RR = {rate_ratio:.3f}  (TÜM arac baseline)')
print(f'Duzeltilmis (dogru): RR = {rr_dogru:.3f}  (SADECE kaza yapan baseline)')
print()

# Yeni Z-test
import math
from scipy.stats import norm
ortak2 = (post_kaza_toplam + len(baseline_kazaref)) / (post_kaza_aracgun + baseline_kazaref_aracgun)
se2 = math.sqrt(ortak2 * (1/post_kaza_aracgun + 1/baseline_kazaref_aracgun))
z2 = (post_kaza_orani - baseline_kazaref_orani) / se2 if se2 > 0 else 0
p2 = 2 * (1 - norm.cdf(abs(z2)))  # iki tarafli (yön bilinmiyor)

print(f'Z-statistic: {z2:.2f}')
print(f'p-value (iki-tarafli): {p2:.6f}')
print()

if p2 < 0.05:
    if rr_dogru > 1:
        print(f'VERIYE GORE: Kaza sonrasi 30g ariza orani DOGRU baseline\'in {rr_dogru:.2f} KATIDIR.')
        print('  -> Kaza sonrasi gercekten ariza firtinasi VAR.')
    elif rr_dogru < 1:
        print(f'VERIYE GORE: Kaza sonrasi 30g ariza orani DOGRU baseline\'in {rr_dogru:.2f} KATI (DAHA AZ).')
        print('  -> Kazadan sonra arac servise giriyor, sefer yapmiyor, ariza fırsati azaliyor.')
        print('  -> "Servis etkisi" mevcut: kaza arac kullanim suresini geciktirir, ariza azalir.')
else:
    print(f'VERIYE GORE: Iki orani arasinda ANLAMLI fark yok (p={p2:.4f}).')
    print('  -> Kaza yapan araclarin kaza-yakin pencere DISINDAKI ariza patternlari benzer.')

# Ek dogrulama: Arac başına RR (kaza yapan her arac kendi baseline'ı ile karsilastirilsin)
print()
print('=== ARAC-BAZLI RR (her arac kendi baseline\'inin esleniği) ===')
arac_rr_list = []
for kapino in list(kaza_yapan_kapinolar)[:500]:  # ornek 500 arac (hizli)
    arac_kazalari = df_kaza[df_kaza['kapino'] == kapino]['kazasaat']
    arac_arizalari = df_ariza[df_ariza['KAPINO'] == kapino]
    if len(arac_arizalari) < 5:
        continue

    # Bu aracin kaza-yakin (post 30g) ve baseline arıza sayisi
    post_n = 0
    post_gun = 0
    for kt in arac_kazalari:
        sub = arac_arizalari[(arac_arizalari['OLAYTARIHI'] > kt) & (arac_arizalari['OLAYTARIHI'] <= kt + timedelta(days=30))]
        post_n += len(sub)
        post_gun += 30

    base_arizalari = arac_arizalari.copy()
    for kt in arac_kazalari:
        base_arizalari = base_arizalari[
            ~((base_arizalari['OLAYTARIHI'] >= kt - timedelta(days=2)) &
              (base_arizalari['OLAYTARIHI'] <= kt + timedelta(days=30)))
        ]
    base_n = len(base_arizalari)
    base_gun = 180 - len(arac_kazalari) * 32  # kaba

    if base_gun > 0 and base_n > 0:
        post_orani_arac = post_n / max(post_gun, 1)
        base_orani_arac = base_n / max(base_gun, 1)
        if base_orani_arac > 0:
            arac_rr_list.append(post_orani_arac / base_orani_arac)

if arac_rr_list:
    arac_rr_arr = np.array(arac_rr_list)
    print(f'500 ornekte arac-bazli RR ortalama: {arac_rr_arr.mean():.3f}')
    print(f'Median: {np.median(arac_rr_arr):.3f}')
    print(f'RR > 1 (firtina var) arac orani: %{(arac_rr_arr > 1).mean()*100:.1f}')
    print(f'RR > 2 (guclu firtina) arac orani: %{(arac_rr_arr > 2).mean()*100:.1f}')


=== METHODOLOJIK DUZELTME ===
Kaza yapan arac sayisi: 1,420

--- BASELINE: Sadece kaza yapan araclarin kaza-olmayan donemleri ---
Ariza sayisi: 17,508
Arac×gun:     192,656
Orani:        0.09088 ariza/arac/gun

--- POST-KAZA: 30 gun sonra ---
Ariza sayisi: 5,267
Arac×gun:     59,010
Orani:        0.08926 ariza/arac/gun

=== ESKI vs DUZELTILMIS RATE RATIO ===
Eski (biased):       RR = 0.951  (TÜM arac baseline)
Duzeltilmis (dogru): RR = 0.982  (SADECE kaza yapan baseline)

Z-statistic: -1.15
p-value (iki-tarafli): 0.252113

VERIYE GORE: Iki orani arasinda ANLAMLI fark yok (p=0.2521).
  -> Kaza yapan araclarin kaza-yakin pencere DISINDAKI ariza patternlari benzer.

=== ARAC-BAZLI RR (her arac kendi baseline'inin esleniği) ===
500 ornekte arac-bazli RR ortalama: 1.239
Median: 0.859
RR > 1 (firtina var) arac orani: %43.0
RR > 2 (guclu firtina) arac orani: %15.6


---

## 15.A. Gun 0 Pre/Post Ayrimi
Gun 0 = x3.83 mixed metrik. Pre-aynigun ve Post-aynigun ayri hesaplanir.

In [16]:
# BOLUM 15.A: Gun 0 Pre/Post Ayrimi
# Cell 11'de gun 0 = x3.83 ama bu mixed (pre+post). Gercek anlik patlamayi olcelim.

gun0 = ariza_kaza_full[ariza_kaza_full['gun_farki_int'] == 0].copy()
gun0['saat_farki_real'] = (gun0['OLAYTARIHI'] - gun0['kazasaat']).dt.total_seconds() / 3600

pre_g0 = gun0[gun0['saat_farki_real'] < 0]
post_g0 = gun0[gun0['saat_farki_real'] > 0]
ayni_an = gun0[gun0['saat_farki_real'] == 0]

print(f'Gun 0 toplam ariza: {len(gun0)}')
print(f'  Kazadan ONCE (ayni gun):  {len(pre_g0)}')
print(f'  Kazadan SONRA (ayni gun): {len(post_g0)}')
print(f'  Tam ayni saat:            {len(ayni_an)}')

# Pre = ortalama 12 saat (kazadan once aynigun = 0.5 gun)
# Post = ortalama 12 saat (kazadan sonra aynigun = 0.5 gun)
pre_aracgun = len(df_kaza) * 0.5
post_aracgun = len(df_kaza) * 0.5

pre_orani = len(pre_g0) / pre_aracgun
post_orani = len(post_g0) / post_aracgun

print()
print('=== AYRI HESAPLANMIS ORANLAR ===')
print(f'Pre-aynigun  (kazadan once 12s):  {pre_orani:.5f}  carpan x{pre_orani/baseline_orani:.2f}')
print(f'Post-aynigun (kazadan sonra 12s): {post_orani:.5f}  carpan x{post_orani/baseline_orani:.2f}')
print(f'Baseline:                          {baseline_orani:.5f}')
print(f'Eski mixed (cell 11):              0.35943   carpan x3.83')
print()
print(f'POST/PRE oran: {post_orani/pre_orani:.2f}x')
print('(Post-aynigun kaza sonrasi anlik fiziksel hasar pencerresi)')

# Cell 11'deki x3.83 yanilticiydi: pre-kaza arizalari da cell 11'deki "kaza gunu" patlamasinda sayildi.
# Gercek post-kaza anlik etki cok daha yuksek (ayri hesaplandi).


Gun 0 toplam ariza: 707
  Kazadan ONCE (ayni gun):  175
  Kazadan SONRA (ayni gun): 532
  Tam ayni saat:            0

=== AYRI HESAPLANMIS ORANLAR ===
Pre-aynigun  (kazadan once 12s):  0.17794  carpan x1.90
Post-aynigun (kazadan sonra 12s): 0.54093  carpan x5.76
Baseline:                          0.09387
Eski mixed (cell 11):              0.35943   carpan x3.83

POST/PRE oran: 3.04x
(Post-aynigun kaza sonrasi anlik fiziksel hasar pencerresi)


---

## 15.B. Confounder Paradoks Aciklamasi
"Kaza yapan az ariza yapiyor" paradoksunun kaynagi: sefer yogunlugu ve servis suresi etkisi.

In [17]:
# BOLUM 15.B: Confounder Paradoks Aciklamasi
# Soru: Neden kaza yapan araclar AZ ariza yapiyor (16.0 vs 17.2)?
# Hipotezler: (1) Sefer yogunlugu farki, (2) Yas dagilimi, (3) Servis süresi etkisi

# 1. SEFER YOGUNLUGU
arac_hatlar_b = pd.read_csv('../panel_data/temiz_veri/arac_gunluk_hatlar.csv')
SEFER_KOL_B = [c for c in arac_hatlar_b.columns if 'SEFER' in c.upper()][0]
arac_sefer_top = arac_hatlar_b.groupby('KAPINO')[SEFER_KOL_B].sum().reset_index()
arac_sefer_top.columns = ['KAPINO', 'toplam_sefer']

kaza_yapan_set = set(df_kaza['kapino'])
arac_sefer_top['kaza_yapti'] = arac_sefer_top['KAPINO'].isin(kaza_yapan_set)

print('=== HIPOTEZ 1: SEFER YOGUNLUGU ===')
sefer_k = arac_sefer_top[arac_sefer_top['kaza_yapti']]['toplam_sefer']
sefer_nk = arac_sefer_top[~arac_sefer_top['kaza_yapti']]['toplam_sefer']
print(f'Kaza yapan ort sefer:    {sefer_k.mean():.0f}  (median {sefer_k.median():.0f})')
print(f'Kaza yapmayan ort sefer: {sefer_nk.mean():.0f}  (median {sefer_nk.median():.0f})')
u1, p1 = stats.mannwhitneyu(sefer_k, sefer_nk, alternative='two-sided')
print(f'Mann-Whitney U: p={p1:.4f}')
if p1 < 0.05:
    yon = 'COK SEFER' if sefer_k.mean() > sefer_nk.mean() else 'AZ SEFER'
    print(f'VERIYE GORE: Kaza yapan araclar {yon} yapiyor.')

# 2. SEFER × ARIZA korelasyonu (genel filo)
arac_ariza_n = df_ariza.groupby('KAPINO').size().reset_index(name='ariza_n')
arac_combined = arac_sefer_top.merge(arac_ariza_n, on='KAPINO', how='inner')
r_sa, p_sa = stats.pearsonr(arac_combined['toplam_sefer'], arac_combined['ariza_n'])
print(f'\nSefer × Ariza korelasyonu (tum filo): r={r_sa:+.3f}  p={p_sa:.4f}')
print('(Eger pozitif → cok sefer = cok ariza, mantikli)')

# 3. SERVIS SURESI MODELI
arac_hatlar_b['TARIH'] = pd.to_datetime(arac_hatlar_b['TARIH'], format='mixed')
print('\n=== HIPOTEZ 3: SERVIS SURESI (kaza sonrasi sefer ara) ===')
servis_gunler = []
for _, row in df_kaza.iterrows():
    kapino = row['kapino']
    kt = row['kazasaat']
    sonraki = arac_hatlar_b[
        (arac_hatlar_b['KAPINO'] == kapino) &
        (arac_hatlar_b['TARIH'].dt.date > kt.date())
    ]
    if len(sonraki) > 0:
        ilk_t = sonraki['TARIH'].min()
        gap = (ilk_t.date() - kt.date()).days
        servis_gunler.append(gap)

if servis_gunler:
    sg = np.array(servis_gunler)
    print(f'Veri var: {len(sg)}/{len(df_kaza)} kaza')
    print(f'Kazadan sonraki ilk sefer gun istatistik:')
    print(f'  Ortalama: {sg.mean():.1f} gun')
    print(f'  Median:   {np.median(sg):.0f} gun')
    print(f'  P25/P75:  {np.percentile(sg,25):.0f}/{np.percentile(sg,75):.0f}')
    print(f'  Max:      {sg.max()} gun')
    print()
    print(f'Kaza ile sonraki sefer arasi:')
    print(f'  Ayni gun ({sg <= 0}):  {(sg <= 0).sum():4d} ({(sg <= 0).mean()*100:.0f}%)')
    print(f'  Ertesi gun:           {(sg == 1).sum():4d} ({(sg == 1).mean()*100:.0f}%)')
    print(f'  >7 gun gap:           {(sg > 7).sum():4d} ({(sg > 7).mean()*100:.0f}%)')
    print(f'  >30 gun gap:          {(sg > 30).sum():4d} ({(sg > 30).mean()*100:.0f}%)')

# SONUC: Paradoks aciklamasi
print()
print('=== PARADOKS YORUMU ===')
print(f'Eger kaza yapan araclar gercekten cok sefer yapiyorsa (r_sefer={r_sa:+.2f}):')
print('  -> Cok sefer = cok ariza ZATEN bekleniyor')
print('  -> Kaza yapan araclarin AZ ariza yapmasi paradoks degil — sefer farki acikliyor olabilir.')
print()
print(f'Eger servis suresi onemli ise (median {np.median(sg) if servis_gunler else "?"} gun gap):')
print('  -> Kazadan sonra arac yolda kalmiyor → ariza fırsati az → 30g ariza orani dusuk goruyor.')


=== HIPOTEZ 1: SEFER YOGUNLUGU ===
Kaza yapan ort sefer:    1344  (median 1370)
Kaza yapmayan ort sefer: 1468  (median 1471)
Mann-Whitney U: p=0.0000
VERIYE GORE: Kaza yapan araclar AZ SEFER yapiyor.

Sefer × Ariza korelasyonu (tum filo): r=-0.128  p=0.0000
(Eger pozitif → cok sefer = cok ariza, mantikli)

=== HIPOTEZ 3: SERVIS SURESI (kaza sonrasi sefer ara) ===
Veri var: 1947/1967 kaza
Kazadan sonraki ilk sefer gun istatistik:
  Ortalama: 1.7 gun
  Median:   1 gun
  P25/P75:  1/2
  Max:      57 gun

Kaza ile sonraki sefer arasi:
  Ayni gun ([False False False ... False False False]):     0 (0%)
  Ertesi gun:           1427 (73%)
  >7 gun gap:             29 (1%)
  >30 gun gap:             8 (0%)

=== PARADOKS YORUMU ===
Eger kaza yapan araclar gercekten cok sefer yapiyorsa (r_sefer=-0.13):
  -> Cok sefer = cok ariza ZATEN bekleniyor
  -> Kaza yapan araclarin AZ ariza yapmasi paradoks degil — sefer farki acikliyor olabilir.

Eger servis suresi onemli ise (median 1.0 gun gap):
  -> Kaz

---

## 15.C. Hava ve Yol Durumu Etkisi
Yagmurlu/kaygan yol kazalari sistem etkilenmesi farkli mi? Kaygan yolda fren haberci mi?

In [18]:
# BOLUM 15.C: Hava ve Yol Durumu Etkisi
# Sorular:
# - Hava/yol durumu kazaya yakin ariza pattern'ini etkiliyor mu?
# - Kaygan yolda fren arizasi pre-kaza yuksek mi?

def post_ariza_say(kaza_subset, gun=30):
    n = 0
    for _, row in kaza_subset.iterrows():
        n += (
            (df_ariza['KAPINO'] == row['kapino']) &
            (df_ariza['OLAYTARIHI'] > row['kazasaat']) &
            (df_ariza['OLAYTARIHI'] <= row['kazasaat'] + timedelta(days=gun))
        ).sum()
    return n

# HAVA
print('=== HAVA DURUMU × POST-30G ARIZA ===')
print(f'{"Hava":15s}  n_kaza  toplam_post  ort_post')
hava_etki = []
for h in df_kaza['hava'].dropna().unique():
    sub = df_kaza[df_kaza['hava'] == h]
    if len(sub) < 20: continue
    post = post_ariza_say(sub)
    ort = post / len(sub)
    hava_etki.append({'hava': h, 'n_kaza': len(sub), 'post': post, 'ort': ort})
    print(f'{str(h)[:15]:15s}  {len(sub):6d}  {post:11d}  {ort:7.2f}')

# YOL
print()
print('=== YOL DURUMU × POST-30G ARIZA ===')
print(f'{"Yol":15s}  n_kaza  toplam_post  ort_post')
for y in df_kaza['yol'].dropna().unique():
    sub = df_kaza[df_kaza['yol'] == y]
    if len(sub) < 20: continue
    post = post_ariza_say(sub)
    ort = post / len(sub)
    print(f'{str(y)[:15]:15s}  {len(sub):6d}  {post:11d}  {ort:7.2f}')

# KAYGAN YOL × FREN HABERCISI MI?
print()
print('=== KAYGAN YOL × FREN ARIZASI 48S HABERCI MI? ===')
kaygan = df_kaza[df_kaza['yol'].isin(['Yağış Kaygan', 'Buz Kaygan'])]
normal = df_kaza[df_kaza['yol'] == 'Normal']

def fren_pre_oran(kaza_subset, kat='FREN ŞİKAYETLERİ', saat=48):
    n_var = 0
    for _, row in kaza_subset.iterrows():
        sub = df_ariza[
            (df_ariza['KAPINO'] == row['kapino']) &
            (df_ariza['OLAYTARIHI'] >= row['kazasaat'] - timedelta(hours=saat)) &
            (df_ariza['OLAYTARIHI'] < row['kazasaat']) &
            (df_ariza['ARIZAUSTKODTANIM'] == kat)
        ]
        if len(sub) > 0: n_var += 1
    return n_var, len(kaza_subset)

f_kaygan_n, f_kaygan_t = fren_pre_oran(kaygan)
f_normal_n, f_normal_t = fren_pre_oran(normal)
print(f'Kaygan yol kazasi: {f_kaygan_t}, 48s once fren arizasi: {f_kaygan_n} (%{f_kaygan_n/max(f_kaygan_t,1)*100:.2f})')
print(f'Normal yol kazasi: {f_normal_t}, 48s once fren arizasi: {f_normal_n} (%{f_normal_n/max(f_normal_t,1)*100:.2f})')

# Chi-square test
import scipy.stats as st
if f_kaygan_t >= 20 and f_normal_t >= 20:
    table = [[f_kaygan_n, f_kaygan_t - f_kaygan_n],
             [f_normal_n, f_normal_t - f_normal_n]]
    chi2, pchi, _, _ = st.chi2_contingency(table)
    print(f'Chi-square: chi2={chi2:.2f}, p={pchi:.4f}')
    if pchi < 0.05:
        print('VERIYE GORE: Kaygan yolda fren arizasinin pre-kaza orani normal yoldan FARKLI (anlamli).')
    else:
        print(f'VERIYE GORE: Kaygan yolda fren arizasinin pre-kaza orani normal yoldan FARKSIZ (p={pchi:.4f}).')


=== HAVA DURUMU × POST-30G ARIZA ===
Hava             n_kaza  toplam_post  ort_post
Akşam               358          943     2.63
Güneşli             699         1761     2.52
Sabah                92          177     1.92
Bilinmiyor          115          449     3.90
Bulutlu             511         1382     2.70
Yağmurlu            152          451     2.97
Karlı                36           91     2.53

=== YOL DURUMU × POST-30G ARIZA ===
Yol              n_kaza  toplam_post  ort_post
Normal             1598         4146     2.59
Bilinmiyor          119          467     3.92
Yağış Kaygan        239          627     2.62

=== KAYGAN YOL × FREN ARIZASI 48S HABERCI MI? ===
Kaygan yol kazasi: 242, 48s once fren arizasi: 5 (%2.07)
Normal yol kazasi: 1598, 48s once fren arizasi: 19 (%1.19)
Chi-square: chi2=0.67, p=0.4141
VERIYE GORE: Kaygan yolda fren arizasinin pre-kaza orani normal yoldan FARKSIZ (p=0.4141).


---

## 15.D. Kaza Siddeti Subgroup
Yaralı yolcu, tek tarafli vs araca carpma, kusur grubu — sistem etkisi farkli mi?

In [19]:
# BOLUM 15.D: Kaza Siddeti Subgroup × Sistem Detayi
# Sorular:
# - Yarali yolcu var/yok karsilastirmasi
# - Tek tarafli (kendi kazasi) vs araca carpma sistem etkilenmesi

def sistem_say(kaza_subset, gun=30, kat=None):
    """Verilen kaza subset'ten post-30g ariza sistem dagilimi."""
    dist = {}
    for _, row in kaza_subset.iterrows():
        sub = df_ariza[
            (df_ariza['KAPINO'] == row['kapino']) &
            (df_ariza['OLAYTARIHI'] > row['kazasaat']) &
            (df_ariza['OLAYTARIHI'] <= row['kazasaat'] + timedelta(days=gun))
        ]
        for k in sub['ARIZAUSTKODTANIM']:
            dist[k] = dist.get(k, 0) + 1
    return dist

# 1. YARALI YOLCU
print('=== YARALI YOLCU × POST-ARIZA ===')
yarali = df_kaza[df_kaza['yaraliyolcu'] > 0]
nyarali = df_kaza[df_kaza['yaraliyolcu'] == 0]
print(f'Yarali olan kaza: {len(yarali)}')
print(f'Yarali olmayan:    {len(nyarali)}')

if len(yarali) >= 5:
    yarali_p = sistem_say(yarali)
    print('Yarali kazalar - post-30g sistem dagilimi:')
    for k, v in sorted(yarali_p.items(), key=lambda x: -x[1])[:5]:
        print(f'  {k}: {v}')
else:
    print('Yarali yolcu sayisi cok az (5 alt) - subgroup analizi yapilamadi.')

# 2. TEK TARAFLI vs ARACA CARPMA
print()
print('=== TEK TARAFLI vs ARACA CARPMA ===')
tek = df_kaza[df_kaza['kategori'].str.contains('Tek Taraflı', na=False)]
ara = df_kaza[df_kaza['kategori'] == 'Araca Çarpma']
print(f'Tek tarafli kaza: {len(tek)}')
print(f'Araca carpma:     {len(ara)}')

if len(tek) >= 20 and len(ara) >= 20:
    tek_d = sistem_say(tek)
    ara_d = sistem_say(ara)
    tek_total = sum(tek_d.values())
    ara_total = sum(ara_d.values())

    print()
    print(f'{"Kategori":35s}  tek_taraf%  araca_carp%  lift')
    tum_kat = set(list(tek_d.keys()) + list(ara_d.keys()))
    karsilastir = []
    for kat in tum_kat:
        tp = tek_d.get(kat, 0) / max(tek_total, 1) * 100
        ap = ara_d.get(kat, 0) / max(ara_total, 1) * 100
        n_tek = tek_d.get(kat, 0)
        if n_tek < 3: continue  # gurultu
        lift = tp / ap if ap > 0 else 0
        karsilastir.append((kat, n_tek, tp, ap, lift))

    karsilastir.sort(key=lambda x: -x[4])
    for kat, n, tp, ap, lift in karsilastir[:10]:
        print(f'{str(kat)[:35]:35s}  {tp:9.2f}  {ap:10.2f}    {lift:.2f}')

# 3. KUSUR GRUBU ETKISI
print()
print('=== KUSUR GRUBU × POST-ARIZA ORTALAMA ===')
for kg in df_kaza['kusurgrubu'].dropna().unique():
    sub = df_kaza[df_kaza['kusurgrubu'] == kg]
    if len(sub) < 10: continue
    n_post = sum(
        ((df_ariza['KAPINO'] == row['kapino']) &
         (df_ariza['OLAYTARIHI'] > row['kazasaat']) &
         (df_ariza['OLAYTARIHI'] <= row['kazasaat'] + timedelta(days=30))).sum()
        for _, row in sub.iterrows()
    )
    print(f'{str(kg)[:30]:30s}  n={len(sub):4d}  ort_post={n_post/len(sub):.2f}')


=== YARALI YOLCU × POST-ARIZA ===
Yarali olan kaza: 1
Yarali olmayan:    1966
Yarali yolcu sayisi cok az (5 alt) - subgroup analizi yapilamadi.

=== TEK TARAFLI vs ARACA CARPMA ===
Tek tarafli kaza: 66
Araca carpma:     1594

Kategori                             tek_taraf%  araca_carp%  lift
KWS SİSTEMİ ARIZALARI                     1.59        0.07    21.98
BASINÇLI YAĞ HATTI ARIZASI                1.59        0.48    3.30
KAYIŞ KASNAK ARIZALARI                    2.38        0.84    2.83
KAPI ARIZALARI                           15.48        6.60    2.35
DİREKSİYON ARIZALARI                      1.59        0.72    2.20
ISITMA SİSTEMİ                            7.94        3.83    2.07
ELEKTRİK SİSTEMİ ARIZALARI               14.29        8.55    1.67
KLİMA SİSTEMİ ARIZALARI                   7.54        5.80    1.30
MOTOR ARIZALARI                           9.92       10.23    0.97
BASINÇLI HAVA DONANIMI ARIZALARI          1.19        1.30    0.92

=== KUSUR GRUBU × POST-ARIZA ORTALA

---

## 15.E. Hat × Kaza Yogunlugu
Egimli hat = daha cok kaza? Analiz 3 ile cross-reference.

In [20]:
# BOLUM 15.E: Hat × Kaza Yogunlugu (Analiz 3 cross-reference)
# Sorular:
# - Hangi hatlarda kaza yogun?
# - Egimli hat = daha cok kaza? (Analiz 3 ile cross-ref)
# - Ayni hat tekrar kaza yapiyor mu?

# 1. Hat basina kaza sayisi
hat_kaza = df_kaza['hatkodu'].value_counts().reset_index()
hat_kaza.columns = ['HATKODU', 'kaza_n']

# 2. Hat egim_puan
with open(r'panel_data\hat_elevation.json', encoding='utf-8') as f:
    hat_elev_raw_e = json.load(f)

def minmax_norm_e(s, clip_q=None):
    if clip_q is not None:
        s = s.clip(upper=s.quantile(clip_q))
    mn, mx = s.min(), s.max()
    return ((s - mn) / (mx - mn) * 100).round(1) if mx > mn else pd.Series(0.0, index=s.index)

hat_elev_e = pd.DataFrame([
    {'HATKODU': k, 'rakim_fark': v.get('rakım_farkı', 0), 'tirmanma_m': v.get('tırmanma_m', 0)}
    for k, v in hat_elev_raw_e.items()
])
hat_elev_e['norm_rakim'] = minmax_norm_e(hat_elev_e['rakim_fark'], clip_q=0.99)
hat_elev_e['norm_tirm'] = minmax_norm_e(hat_elev_e['tirmanma_m'], clip_q=0.99)
hat_elev_e['egim_puan'] = (hat_elev_e['norm_rakim'] * 0.4 + hat_elev_e['norm_tirm'] * 0.6).round(1)

# 3. Hat sefer sayisi (normalize icin)
hat_sefer_e = arac_hatlar_b.groupby('HATKODU')[SEFER_KOL_B].sum().reset_index()
hat_sefer_e.columns = ['HATKODU', 'toplam_sefer']

hat_combine = hat_kaza.merge(hat_elev_e[['HATKODU', 'egim_puan']], on='HATKODU', how='left')
hat_combine = hat_combine.merge(hat_sefer_e, on='HATKODU', how='left')
hat_combine['kaza_per_1000_sefer'] = (
    hat_combine['kaza_n'] / hat_combine['toplam_sefer'] * 1000
).round(3)
hat_combine = hat_combine.dropna(subset=['egim_puan', 'toplam_sefer'])

# 4. En cok kaza yapan hatlar
print('=== EN COK KAZA OLAN 15 HAT ===')
print(hat_combine.nlargest(15, 'kaza_n')[['HATKODU','kaza_n','egim_puan','toplam_sefer','kaza_per_1000_sefer']].to_string(index=False))

# 5. Sefer basina en cok kaza yapan hatlar (kucuk hatlar dahil)
print()
print('=== SEFER BASINA EN COK KAZA HATLAR (min 100 sefer, 3 kaza) ===')
hat_t = hat_combine[(hat_combine['toplam_sefer'] >= 100) & (hat_combine['kaza_n'] >= 3)]
print(hat_t.nlargest(15, 'kaza_per_1000_sefer')[['HATKODU','kaza_n','egim_puan','toplam_sefer','kaza_per_1000_sefer']].to_string(index=False))

# 6. Egim × Kaza korelasyonu
print()
print('=== EGIM × KAZA KORELASYONU ===')
print(f'Test edilen hat: {len(hat_t)}')
r1, p1 = stats.pearsonr(hat_t['egim_puan'], hat_t['kaza_n'])
r2, p2 = stats.pearsonr(hat_t['egim_puan'], hat_t['kaza_per_1000_sefer'])
print(f'Egim × kaza_n:                 r={r1:+.3f}  p={p1:.4f}')
print(f'Egim × kaza_per_1000_sefer:    r={r2:+.3f}  p={p2:.4f}')

if p2 < 0.05:
    yon = 'POZITIF' if r2 > 0 else 'NEGATIF'
    print(f'VERIYE GORE: Egim ile kaza yogunlugu arasinda {yon} ANLAMLI iliski (p<0.05).')
else:
    print(f'VERIYE GORE: Egim ile kaza yogunlugu arasinda anlamli iliski YOK (p={p2:.4f}).')

# 7. Tekrar eden hatlar
print()
print('=== AYNI HATLARDA TEKRAR KAZA ===')
print(f'Toplam hat: {len(hat_kaza)}')
print(f'1 kaza:  {(hat_kaza["kaza_n"] == 1).sum()}')
print(f'2-5 kaza: {((hat_kaza["kaza_n"] >= 2) & (hat_kaza["kaza_n"] <= 5)).sum()}')
print(f'6-10 kaza: {((hat_kaza["kaza_n"] >= 6) & (hat_kaza["kaza_n"] <= 10)).sum()}')
print(f'10+ kaza: {(hat_kaza["kaza_n"] >= 10).sum()}')

# Tekrar kaza yapan hatlarin egim profili
ust_hatlar = hat_combine[hat_combine['kaza_n'] >= 10]
if len(ust_hatlar) >= 5:
    print(f'\nCok kaza yapan ({len(ust_hatlar)} hat) ort egim_puan: {ust_hatlar["egim_puan"].mean():.1f}')
    print(f'Az kaza yapan (sadece 1 kaza) ort egim_puan: {hat_combine[hat_combine["kaza_n"]==1]["egim_puan"].mean():.1f}')


=== EN COK KAZA OLAN 15 HAT ===
HATKODU  kaza_n  egim_puan  toplam_sefer  kaza_per_1000_sefer
    34G      44       66.9        239756                0.184
   146T      32       74.1         31403                1.019
   142F      31       30.8         45980                0.674
    147      30       59.1         28495                1.053
   HT29      27       64.1         25884                1.043
   34BZ      26       50.6        302634                0.086
   142B      24       71.3         30732                0.781
    146      22       66.5         23387                0.941
    19S      21       51.3         40140                0.523
    98G      20       42.6          9330                2.144
   132V      20       66.8         33769                0.592
   34AS      19       44.0        312981                0.061
    18Ü      19       52.4         27293                0.696
    89C      18       49.9         15580                1.155
     10      17       43.3         455